# COPY INTO and MERGE Commands

## COPY INTO command
- Incrementally loads data into Delta Lake tables from Cloud Storage
- Supports schema evolution
- Supports wide range of file formats (CSV, JSON, Parquet, Delta)
- Alternative to Auto Loader for batch ingestion

### Create the table to copy the data into

In [0]:
DROP TABLE IF EXISTS demo.delta_lake.raw_stock_prices;

In [0]:
CREATE TABLE IF NOT EXISTS demo.delta_lake.raw_stock_prices;

### Incrementally load new files into the table

In [0]:
DELETE FROM demo.delta_lake.raw_stock_prices;

COPY INTO demo.delta_lake.raw_stock_prices
FROM 'abfss://demo@philproject1.dfs.core.windows.net/landing/stock_prices'
FILEFORMAT = JSON
FORMAT_OPTIONS('inferschema' = 'True')
COPY_OPTIONS('mergeSchema' = 'True') 

num_affected_rows,num_inserted_rows,num_skipped_corrupt_files
6,6,0


In [0]:
SELECT * FROM demo.delta_lake.raw_stock_prices

price,status,stock_id,trading_date
233.5,ACTIVE,AAPL,2025-02-12
2805.0,ACTIVE,GOOGL,2025-02-12
335.0,ACTIVE,MSFT,2025-02-12
3520.0,ACTIVE,AMZN,2025-02-12
405.0,ACTIVE,META,2025-02-12
null,DELISTED,TSLA,2025-02-12


## MERGE Statement
- Used for upserts (Insert/Update/Delete operations in a single statement)
- Allows merging new data into a target table based on matching condition

### Create the table to merge the data into

In [0]:
DROP TABLE IF EXISTS demo.delta_lake.stock_prices;
CREATE TABLE IF NOT EXISTS demo.delta_lake.stock_prices
(
  stock_id STRING,
  price DOUBLE,
  trading_date DATE 
)

### Merge the source data into target table
1. Insert new stocks received
2. Update price and trading_date if updates received
3. Delete stocks which are de-listed from the exchanmmge (status = 'DELISTED')

In [0]:
MERGE INTO demo.delta_lake.stock_prices AS target
USING demo.delta_lake.raw_stock_prices AS source
  ON target.stock_id = source.stock_id
WHEN MATCHED AND source.status = 'ACTIVE' THEN
  UPDATE SET target.price = source.price, target.trading_date = source.trading_date
WHEN MATCHED AND source.status = 'DELISTED' THEN
  DELETE
WHEN NOT MATCHED AND source.status = 'ACTIVE' THEN 
  INSERT (stock_id, price, trading_date) VALUES (source.stock_id,source.price, source.trading_date)

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
6,4,1,1


In [0]:
SELECT * FROM demo.delta_lake.stock_prices

stock_id,price,trading_date
AAPL,233.5,2025-02-12
GOOGL,2805.0,2025-02-12
MSFT,335.0,2025-02-12
AMZN,3520.0,2025-02-12
META,405.0,2025-02-12
